# Encoder–HistGB: out-of-fold predictions

The baseline uses the same pipeline from **features_selection.ipynb**:
current imputation, engineered features and tuned HistGB, without feature selection.
HistGB receives its existing features plus only the encoder's predicted OFF score per visit.

The encoder receives the original imputed features, their missingness indicators and elapsed
time since the first visit. It uses the best confirmed parameters from
**opt.ipynb** (trial 19): three bidirectional Transformer blocks, eight heads,
width 32 and no CLS token. Each patient's available visits form one sequence.

Within each outer training split, five patient-separated inner folds produce one out-of-fold
encoder prediction for every training row. HistGB trains on the entire outer training split.
The encoder is then refitted on that entire split to predict the outer validation patients.
Each encoder fit includes its own preprocessing and internal patient holdout for epoch selection.
The outer validation patients train neither model.


## 1. Reuse the exact baseline and set training options

Only the five setup cells from features_selection.ipynb are executed, not its validation or
feature-selection experiments. Paths work from the project root, exploration or stacking folder.
PyTorch is required: uv pip install --python .venv/bin/python torch.

In [7]:
import json
import os
from pathlib import Path
from copy import deepcopy
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold

PROJECT_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / "exploration/features_selection.ipynb").is_file())
reference = json.loads((PROJECT_DIR / "exploration/features_selection.ipynb").read_text())
previous_dir = Path.cwd()
try:
    os.chdir(PROJECT_DIR)  # The reference setup reads data relative to the project.
    for i in [1, 3, 4, 6, 8]:  # Imports/data, history, imputation, features, pipeline.
        exec("".join(reference["cells"][i]["source"]))
finally:
    os.chdir(previous_dir)
DATA_DIR = PROJECT_DIR / "data"
baseline = clone(pipeline)

# Winning confirmation configuration (trial 19) from opt.ipynb.
ENCODER = dict(width=32, heads=8, layers=3, dropout=0.1347886862599206,
               ff_multiplier=3, mlp_layers=2, mlp_units=16)
MAX_EPOCHS, PATIENCE, BATCH_SIZE = 100, 10, 128
LEARNING_RATE, WEIGHT_DECAY = 0.001759366313629433, 4.3514588969246546e-05
SEEDS = [42, 43, 44]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")
torch.set_num_threads(3)
print(f"Device: {DEVICE}; baseline: {baseline[-1]}")

Device: mps; baseline: HistGradientBoostingRegressor(l2_regularization=1.8115270802885095,
                              learning_rate=0.042592090635273853,
                              max_features=0.6078232265551354, max_iter=800,
                              max_leaf_nodes=62, min_samples_leaf=25,
                              random_state=42)


## 2. One sequence per patient

Patient IDs only group visits; they are never encoder inputs. Elapsed time encodes the actual
spacing between visits. Padding uses row number -1, which is ignored by attention and MSE.
The row numbers also restore predictions to the original visit order after batching.

In [8]:
def encoder_features(imputed):
    features = imputed.drop(columns="patient_id").copy()
    features["time_since_first_visit"] = (
        imputed["age"] - imputed.groupby("patient_id")["age"].transform("min")
    )
    return features


def patient_sequences(raw, features, targets=None):
    visits = raw.reset_index(drop=True).sort_values(["patient_id", "age"])
    rows = [torch.tensor(group.index.to_numpy())
            for _, group in visits.groupby("patient_id", sort=False)]
    values = torch.tensor(np.asarray(features), dtype=torch.float32)
    labels = torch.tensor(np.zeros(len(raw)) if targets is None else np.asarray(targets),
                          dtype=torch.float32)
    return TensorDataset(
        pad_sequence([values[i] for i in rows], batch_first=True),
        pad_sequence([labels[i] for i in rows], batch_first=True),
        pad_sequence(rows, batch_first=True, padding_value=-1),
    )

## 3. Bidirectional encoder

Dense input projection → three sequential TransformerEncoder layers → one embedding per visit.
Each layer already includes parallel attention heads, feed-forward layers, residual connections
and LayerNorm. The regression head has two hidden layers and supplies the extra HistGB feature.


In [9]:
class VisitEncoder(nn.Module):
    def __init__(self, n_features, width=64, heads=4, layers=2, dropout=0.1,
                 ff_multiplier=2, mlp_layers=0, mlp_units=32):
        super().__init__()
        self.projection = nn.Linear(n_features, width)
        layer = nn.TransformerEncoderLayer(
            d_model=width, nhead=heads, dim_feedforward=ff_multiplier * width,
            dropout=dropout, activation="gelu", batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers,
                                             enable_nested_tensor=False)
        # TransformerEncoder copies its layer: give each block independent weights.
        for parameter in self.encoder.parameters():
            if parameter.ndim > 1:
                nn.init.xavier_uniform_(parameter)
        head, input_dim = [], width
        for _ in range(mlp_layers):
            head.extend([nn.Linear(input_dim, mlp_units), nn.GELU(), nn.Dropout(dropout)])
            input_dim = mlp_units
        self.head = nn.Sequential(*head, nn.Linear(input_dim, 1))

    def encode(self, values, padding):
        return self.encoder(self.projection(values), src_key_padding_mask=padding,
                            is_causal=False)

    def forward(self, values, padding):
        return self.head(self.encode(values, padding)).squeeze(-1)

## 4. Train the encoder inside a training split

Both input and target scaling are fitted on training patients only. Target scaling improves
optimization; reported predictions and MSE are converted back to OFF score units.
Early stopping uses its own patient holdout, including independently fitted preprocessing.
The selected epoch count is then used to refit a fresh encoder on its entire training split.

In [10]:
def train_network(training, seed, epochs, validation=None):
    torch.manual_seed(seed)
    network = VisitEncoder(training.tensors[0].shape[-1], **ENCODER).to(DEVICE)
    optimizer = torch.optim.AdamW(network.parameters(), lr=LEARNING_RATE,
                                 weight_decay=WEIGHT_DECAY)
    loader = DataLoader(training, batch_size=BATCH_SIZE, shuffle=True)
    best_loss, best_epoch, best_state, stale = np.inf, epochs, None, 0
    history = []

    for epoch in range(1, epochs + 1):
        network.train()
        total_error, count = 0.0, 0
        for values, targets, rows in loader:
            values, targets, rows = (t.to(DEVICE) for t in (values, targets, rows))
            valid = rows >= 0
            optimizer.zero_grad()
            prediction = network(values, ~valid)
            loss = nn.functional.mse_loss(prediction[valid], targets[valid])
            loss.backward()
            nn.utils.clip_grad_norm_(network.parameters(), 1.0)
            optimizer.step()
            total_error += loss.item() * valid.sum().item()
            count += valid.sum().item()
        history.append({"epoch": epoch, "train_MSE": total_error / count})

        if validation is not None:
            network.eval()
            total_error, count = 0.0, 0
            with torch.no_grad():
                for values, targets, rows in DataLoader(validation, batch_size=BATCH_SIZE):
                    values, targets, rows = (t.to(DEVICE) for t in (values, targets, rows))
                    valid = rows >= 0
                    prediction = network(values, ~valid)
                    total_error += ((prediction[valid] - targets[valid]) ** 2).sum().item()
                    count += valid.sum().item()
            stop_loss = total_error / count
            history[-1]["stop_MSE"] = stop_loss
            if stop_loss < best_loss:
                best_loss, best_epoch, stale = stop_loss, epoch, 0
                best_state = deepcopy(network.state_dict())
            else:
                stale += 1
            if stale >= PATIENCE:
                break

    if best_state is not None:
        network.load_state_dict(best_state)
    return network.eval(), best_epoch, pd.DataFrame(history)


def fit_encoder(raw, targets, imputed, seed):
    fit_idx, stop_idx = next(GroupKFold(
        n_splits=5, shuffle=True, random_state=seed
    ).split(raw, groups=raw["patient_id"]))
    fit_raw, stop_raw = raw.iloc[fit_idx], raw.iloc[stop_idx]
    inner_prep = clone(baseline[:2])
    fit_features = encoder_features(inner_prep.fit_transform(fit_raw))
    stop_features = encoder_features(inner_prep.transform(stop_raw))
    scaler = StandardScaler().fit(fit_features)
    mean = targets.iloc[fit_idx].mean()
    scale = max(targets.iloc[fit_idx].std(ddof=0), 1e-8)
    training = patient_sequences(fit_raw, scaler.transform(fit_features),
                                 (targets.iloc[fit_idx] - mean) / scale)
    stopping = patient_sequences(stop_raw, scaler.transform(stop_features),
                                 (targets.iloc[stop_idx] - mean) / scale)
    _, epochs, history = train_network(training, seed, MAX_EPOCHS, stopping)
    history[["train_MSE", "stop_MSE"]] *= scale ** 2

    # Fresh model and scaling, now fitted on every patient in this encoder-training split.
    features = encoder_features(imputed)
    scaler = StandardScaler().fit(features)
    mean, scale = targets.mean(), max(targets.std(ddof=0), 1e-8)
    training = patient_sequences(raw, scaler.transform(features), (targets - mean) / scale)
    network, _, _ = train_network(training, seed, epochs)
    network.requires_grad_(False)
    return dict(network=network, scaler=scaler, target_mean=mean, target_scale=scale,
                epochs=epochs, history=history)

## 5. Extract frozen encoder predictions in the original row order

HistGB receives its existing features plus the encoder's prediction in original OFF score units.
Training predictions are out of fold; validation predictions come from the encoder refitted on
all outer training patients. Embeddings remain internal to the encoder.


In [11]:
@torch.no_grad()
def encoder_outputs(fitted, raw, imputed):
    features = fitted["scaler"].transform(encoder_features(imputed))
    sequences = patient_sequences(raw, features)
    network = fitted["network"].eval()
    predictions = np.empty(len(raw), dtype=np.float32)
    for values, _, rows in DataLoader(sequences, batch_size=BATCH_SIZE):
        padding = rows < 0
        prediction = network(values.to(DEVICE), padding.to(DEVICE)).cpu()
        valid_rows = rows[~padding].numpy()
        predictions[valid_rows] = prediction[~padding].numpy()
    predictions = predictions * fitted["target_scale"] + fitted["target_mean"]
    return pd.Series(predictions, index=raw.index, name="encoder_prediction")


## 6. Fold-specific Encoder–HistGB validation

Use the same five patient-separated outer splits as the existing notebooks. Within each outer
training split, five patient-separated inner folds generate encoder predictions: fit the encoder
and its preprocessing on four folds, then predict the held-out fold. Every training row receives
one out-of-fold prediction. Keep these inner folds fixed across seeds.

Fit HistGB on all outer training rows using its existing features plus this prediction column.
Refit an encoder on all outer training patients for the validation prediction column. The baseline
also uses all outer training rows and the same HistGB parameters. Outer validation patients never
fit preprocessing, either model or early stopping.

Negative delta_MSE means the encoder prediction helps. The standard deviation describes variability
across splits and seeds, not uncertainty about the true test performance.


In [12]:
cv = GroupKFold(n_splits=5, shuffle=True, random_state=42)
oof_cv = GroupKFold(n_splits=5)
scores = []
with threadpool_limits(limits=3):
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y, groups=X["patient_id"]), 1):
        raw_train, raw_val = X.iloc[train_idx], X.iloc[val_idx]
        yt, yv = y.iloc[train_idx], y.iloc[val_idx]
        prep = clone(baseline[:-1])
        train = prep.fit_transform(raw_train, yt)
        validation = prep.transform(raw_val)
        imputed_train = prep[:2].transform(raw_train)
        imputed_val = prep[:2].transform(raw_val)
        original = clone(baseline[-1]).fit(train, yt)
        baseline_mse = mean_squared_error(yv, original.predict(validation))

        for seed in SEEDS:
            train_prediction = pd.Series(index=raw_train.index, dtype=float,
                                         name="encoder_prediction")
            for encoder_idx, oof_idx in oof_cv.split(raw_train, yt, groups=raw_train["patient_id"]):
                raw_encoder, raw_oof = raw_train.iloc[encoder_idx], raw_train.iloc[oof_idx]
                encoder_prep = clone(baseline[:2])
                imputed_encoder = encoder_prep.fit_transform(raw_encoder)
                imputed_oof = encoder_prep.transform(raw_oof)
                fitted_encoder = fit_encoder(raw_encoder, yt.iloc[encoder_idx], imputed_encoder, seed)
                train_prediction.iloc[oof_idx] = encoder_outputs(
                    fitted_encoder, raw_oof, imputed_oof
                ).to_numpy()
            fitted_encoder = fit_encoder(raw_train, yt, imputed_train, seed)
            encoder_prediction = encoder_outputs(fitted_encoder, raw_val, imputed_val)
            augmented = clone(baseline[-1]).fit(train.join(train_prediction), yt)
            mse = mean_squared_error(yv, augmented.predict(validation.join(encoder_prediction)))
            scores.append(dict(fold=fold, seed=seed, epochs=fitted_encoder["epochs"],
                               HistGB=baseline_mse, Transformer=mean_squared_error(yv, encoder_prediction),
                               Encoder_HistGB=mse, delta_MSE=mse - baseline_mse))
            print(f"Fold {fold}/5, seed {seed}, {fitted_encoder['epochs']} epochs: "
                  f"HistGB {baseline_mse:.4f} → Encoder–HistGB {mse:.4f}")
scores = pd.DataFrame(scores)
scores

Fold 1/5, seed 42, 56 epochs: HistGB 11.9597 → Encoder–HistGB 8.6907
Fold 1/5, seed 43, 79 epochs: HistGB 11.9597 → Encoder–HistGB 7.7566
Fold 1/5, seed 44, 66 epochs: HistGB 11.9597 → Encoder–HistGB 7.7468
Fold 2/5, seed 42, 55 epochs: HistGB 11.7222 → Encoder–HistGB 8.3625
Fold 2/5, seed 43, 55 epochs: HistGB 11.7222 → Encoder–HistGB 7.8223
Fold 2/5, seed 44, 79 epochs: HistGB 11.7222 → Encoder–HistGB 8.1123
Fold 3/5, seed 42, 51 epochs: HistGB 11.2770 → Encoder–HistGB 8.1228
Fold 3/5, seed 43, 60 epochs: HistGB 11.2770 → Encoder–HistGB 8.0888
Fold 3/5, seed 44, 48 epochs: HistGB 11.2770 → Encoder–HistGB 8.8274
Fold 4/5, seed 42, 56 epochs: HistGB 12.1577 → Encoder–HistGB 8.3349
Fold 4/5, seed 43, 53 epochs: HistGB 12.1577 → Encoder–HistGB 8.5790
Fold 4/5, seed 44, 18 epochs: HistGB 12.1577 → Encoder–HistGB 9.0724
Fold 5/5, seed 42, 35 epochs: HistGB 11.5505 → Encoder–HistGB 8.3941
Fold 5/5, seed 43, 60 epochs: HistGB 11.5505 → Encoder–HistGB 7.8549
Fold 5/5, seed 44, 58 epochs: Hist

,fold,seed,epochs,HistGB,Transformer,Encoder_HistGB,delta_MSE
0,1,42,56,11.959653,8.763222,8.690736,-3.268917
1,1,43,79,11.959653,7.725109,7.756601,-4.203052
2,1,44,66,11.959653,7.772806,7.746830,-4.212823
3,2,42,55,11.722212,8.763608,8.362477,-3.359735
4,2,43,55,11.722212,8.196905,7.822277,-3.899934
5,2,44,79,11.722212,8.610436,8.112285,-3.609927
6,3,42,51,11.277018,8.281631,8.122835,-3.154183
7,3,43,60,11.277018,8.412965,8.088834,-3.188183
8,3,44,48,11.277018,9.595924,8.827449,-2.449569
9,4,42,56,12.157750,8.196305,8.334918,-3.822832


In [13]:
summary = scores[["HistGB", "Transformer", "Encoder_HistGB"]].agg(
    ["mean", lambda s: s.std(ddof=0)]
).T
summary.columns = ["MSE", "std"]
summary["RMSE"] = np.sqrt(summary["MSE"])
summary["gain_MSE_pct"] = 100 * (1 - summary["MSE"] / summary.loc["HistGB", "MSE"])
summary

,MSE,std,RMSE,gain_MSE_pct
HistGB,11.733420,0.307616,3.425408,0.000000
Transformer,8.534545,0.553791,2.921394,27.262943
Encoder_HistGB,8.305371,0.410118,2.881904,29.216114


## Alternatives to test later

- Add the existing engineered features to encoder inputs.
- Vary encoder depth, width and heads, or replace the encoder architecture.
- Predict ON/OFF measurements with this encoder or a separate encoder.
